In [ ]:
# scikit-learn                   1.3.2         py39h172c841_2            conda-forge
# pytorch                        2.3.0         gpu_mps_py39hdbe6d3a_101  pkgs/main 
# python                         3.9.23        h7139b31_0_cpython        conda-forge
# numpy-indexed                  0.3.7         pyhd8ed1ab_0              conda-forge
# numpy                          1.26.4        py39h901140f_1            pkgs/main
# jupyterlab                     4.4.7         py39hca03da5_0            pkgs/main 
# ipykernel                      6.30.1        pyh92f572d_0              conda-forge
# dgl                            1.1.3         pypi_0                    pypi 

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import pickle

import utils
from model import MyModel

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

torch: 2.3.0.post101
cuda available: False


In [2]:
YEAR = 2015
NUM_HIDDEN_LAYERS = 1
EMBEDDING_SIZE = 128
MULTITASK_WEIGHTS = (0.5, 0.25, 0.25)

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
device = torch.device(DEVICE)

print("Using device:", device)
print("Config:", YEAR, NUM_HIDDEN_LAYERS, EMBEDDING_SIZE, MULTITASK_WEIGHTS)

Using device: cpu
Config: 2015 1 128 (0.5, 0.25, 0.25)


In [3]:
data = utils.load_dataset(year=YEAR)

g = utils.build_graph_from_matrix(
    data["ct_adjacency_withweight"],
    data["node_feats"].astype(np.float32),
    device=DEVICE
).to(device)

distm = data["distm"]  # distance/time matrix used in training

print("num_nodes:", data["num_nodes"])
print("node_feats shape:", data["node_feats"].shape)
print("distm shape:", distm.shape)

/Users/kazishahrukhomar/Documents/MISC/GMEL-master/code/utils.py:48: UserWarning: Feature table contains NaN. 0 is used to fill these NaNs
  warnings.warn('Feature table contains NaN. 0 is used to fill these NaNs')


num_nodes: 2168
node_feats shape: (2168, 65)
distm shape: (2168, 2168)


/Users/kazishahrukhomar/Documents/MISC/GMEL-master/envs/lib/python3.9/site-packages/dgl/heterograph.py:92: DGLWarning: Recommend creating graphs by `dgl.graph(data)` instead of `dgl.DGLGraph(data)`.
  dgl_warning(
/Users/kazishahrukhomar/Documents/MISC/GMEL-master/code/utils.py:136: RuntimeWarning: divide by zero encountered in divide
  norm = 1.0 / in_deg


In [24]:
data['test'][2000]

array([25, 19, 23])

In [13]:
# gbrt_path = f"./models/gbrt_year{YEAR}_layers{NUM_HIDDEN_LAYERS}_emb{EMBEDDING_SIZE}_multitask{MULTITASK_WEIGHTS}.txt"
# assert os.path.exists(gbrt_path), f"GBRT model not found: {gbrt_path}"

# with open(gbrt_path, "rb") as f:
#     gbrt = pickle.load(f)

# print("Loaded GBRT:", type(gbrt))

In [5]:
import sklearn.ensemble._gb as _gb  # internal module containing GradientBoostingRegressor

class SklearnCompatUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        # Old sklearn path used in legacy pickles
        if module == "sklearn.ensemble.gradient_boosting":
            module = "sklearn.ensemble._gb"
        return super().find_class(module, name)

def load_legacy_gbrt(path):
    with open(path, "rb") as f:
        return SklearnCompatUnpickler(f).load()

In [6]:
gbrt_path = f"./models/gbrt_year{YEAR}_layers{NUM_HIDDEN_LAYERS}_emb{EMBEDDING_SIZE}_multitask{MULTITASK_WEIGHTS}.txt"
assert os.path.exists(gbrt_path), f"GBRT model not found: {gbrt_path}"

gbrt = load_legacy_gbrt(gbrt_path)

print("Loaded GBRT:", type(gbrt))

AttributeError: Can't get attribute 'LeastSquaresError' on <module 'sklearn.ensemble._gb' from '/Users/kazishahrukhomar/Documents/MISC/GMEL-master/envs/lib/python3.9/site-packages/sklearn/ensemble/_gb.py'>

In [7]:
import numpy as np
import pickle
from sklearn.ensemble import GradientBoostingRegressor
import utils
import os

YEAR = 2015
L = 1
E = 128
W = (0.5, 0.25, 0.25)

data = utils.load_dataset(year=YEAR)
train_data = data["train"]
distm = data["distm"]

emb_path = f"./embeddings/censustract_embeddings_year{YEAR}_layers{L}_emb{E}_multitask{W}.npz"
assert os.path.exists(emb_path), f"Embeddings not found: {emb_path}"

z = np.load(emb_path)
src_emb, dst_emb = z["arr_0"], z["arr_1"]

scaled_distm = distm / distm.max() * np.max([src_emb.max(), dst_emb.max()])

def construct_feat(triplets):
    feat_src = src_emb[triplets[:, 0]]
    feat_dst = dst_emb[triplets[:, 1]]
    feat_dist = scaled_distm[triplets[:, 0], triplets[:, 1]].reshape(-1, 1)
    X = np.concatenate([feat_src, feat_dst, feat_dist], axis=1)
    y = triplets[:, 2]
    return X, y

X_train, y_train = construct_feat(train_data)

gbrt = GradientBoostingRegressor(max_depth=2, random_state=2019, n_estimators=100)
gbrt.fit(X_train, y_train)

new_path = f"./models/gbrt_year{YEAR}_layers{L}_emb{E}_multitask{W}_sklearn132.pkl"
with open(new_path, "wb") as f:
    pickle.dump(gbrt, f)

print("Saved new GBRT:", new_path)

/Users/kazishahrukhomar/Documents/MISC/GMEL-master/code/utils.py:48: UserWarning: Feature table contains NaN. 0 is used to fill these NaNs
  warnings.warn('Feature table contains NaN. 0 is used to fill these NaNs')


Saved new GBRT: ./models/gbrt_year2015_layers1_emb128_multitask(0.5, 0.25, 0.25)_sklearn132.pkl


In [8]:
def get_embeddings(year, L, E, W, g, device):
    emb_path = f"./embeddings/censustract_embeddings_year{year}_layers{L}_emb{E}_multitask{W}.npz"
    if os.path.exists(emb_path):
        z = np.load(emb_path)
        src_emb, dst_emb = z["arr_0"], z["arr_1"]
        print("Loaded embeddings from:", emb_path, src_emb.shape, dst_emb.shape)
        return src_emb, dst_emb

    # fallback: compute from GMEL checkpoint
    ckpt_path = f"./models/model_state_layers{L}_emb{E}_multitask{W}.pth"
    assert os.path.exists(ckpt_path), f"Checkpoint not found: {ckpt_path}"

    model = MyModel(
        g=g,
        num_nodes=g.number_of_nodes(),
        in_dim=g.ndata["attr"].shape[1],
        h_dim=E,
        num_hidden_layers=L,
        device=str(device),
        reg_param=0
    ).to(device)

    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()

    with torch.no_grad():
        src_emb = model(g).detach().cpu().numpy()
        dst_emb = model.forward2(g).detach().cpu().numpy()

    print("Computed embeddings from checkpoint:", ckpt_path, src_emb.shape, dst_emb.shape)
    return src_emb, dst_emb


src_emb, dst_emb = get_embeddings(YEAR, NUM_HIDDEN_LAYERS, EMBEDDING_SIZE, MULTITASK_WEIGHTS, g, device)

Loaded embeddings from: ./embeddings/censustract_embeddings_year2015_layers1_emb128_multitask(0.5, 0.25, 0.25).npz (2168, 128) (2168, 128)


In [9]:
def scale_dist_like_training(distm, src_emb, dst_emb):
    return distm / distm.max() * np.max([src_emb.max(), dst_emb.max()])

scaled_distm = scale_dist_like_training(distm, src_emb, dst_emb)

def make_features_from_pairs(pairs_nodeid, src_emb, dst_emb, scaled_distm):
    """
    pairs_nodeid: np.array shape (K,2) with node_id indices [src, dst]
    """
    pairs_nodeid = np.asarray(pairs_nodeid, dtype=int)
    feat_src = src_emb[pairs_nodeid[:, 0]]
    feat_dst = dst_emb[pairs_nodeid[:, 1]]
    feat_dist = scaled_distm[pairs_nodeid[:, 0], pairs_nodeid[:, 1]].reshape(-1, 1)
    X = np.concatenate([feat_src, feat_dst, feat_dist], axis=1)
    return X

print("Example feature dim:", make_features_from_pairs([[0,1]], src_emb, dst_emb, scaled_distm).shape)

Example feature dim: (1, 257)


In [25]:
def predict_pairs_nodeid(pairs_nodeid, gbrt, src_emb, dst_emb, scaled_distm):
    X = make_features_from_pairs(pairs_nodeid, src_emb, dst_emb, scaled_distm)
    y_hat = gbrt.predict(X)
    return y_hat

# Example: pick any valid node ids
pairs = np.array([[25, 19]], dtype=int)

y_hat = predict_pairs_nodeid(pairs, gbrt, src_emb, dst_emb, scaled_distm)
print("Pred:", list(zip(pairs.tolist(), y_hat.tolist())))

Pred: [([25, 19], 29.82793650419287)]


In [19]:
pairs

array([[ 10, 500]])